# Association Rule

Dữ liệu giao thông có số lượng "Items" (các tuyến đường) rất lớn. Do đó, thay vì thuật toán Apriori truyền thống, nhóm ưu tiên sử dụng FP-Growth vì tốc độ tính toán ma trận thưa (sparse matrix) của nó vượt trội hơn.

### Bước 1: Định nghĩa Giỏ hàng (Transaction) và Mặt hàng (Item)

Chỉ tập trung vào các tình trạng giao thông xấu để tìm quy luật kẹt xe.



In [1]:
from datetime import datetime, timezone
now = datetime.now(timezone.utc)
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
import unicodedata

# 0. Nạp dữ liệu từ file nén của hệ thống mới và xử lý nhiễu
df_compact = pd.read_csv("train_built_compact.csv")
df_compact = df_compact.dropna(subset=['street_name'])

# 1. Lọc các dòng có mức độ kẹt xe cao (LOS là E hoặc F)
df_congestion = df_compact[df_compact['LOS'].isin(['E', 'F'])].copy()

# 2. Sử dụng biến day_type thay cho is_weekend để lọc ngày thường
df_weekday_congestion = df_congestion[df_congestion['day_type'] == 'weekday'].copy()

# 3. Đồng bộ unicode Tiếng Việt (Đưa tất cả về chuẩn NFC)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].apply(lambda x: unicodedata.normalize('NFC', str(x)))

# 4. Làm sạch text (tránh trùng lặp tên đường)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].str.replace(r'\s+', ' ', regex=True)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].str.replace(r'^Đường\s+', '', regex=True)
df_weekday_congestion['street_name'] = df_weekday_congestion['street_name'].str.strip()

# 5. Gom giỏ hàng
transactions = df_weekday_congestion.groupby(['date', 'time_slot'])['street_name'].apply(lambda x: list(set(x))).tolist()
transactions = [t for t in transactions if len(t) > 1]
print(f"Số lượng Transactions (Khung giờ kẹt nhiều đường): {len(transactions):,}")
print(f"Mẫu 3 Transactions (Đã lọc trùng): {transactions[:3]}")

Số lượng Transactions (Khung giờ kẹt nhiều đường): 289
Mẫu 3 Transactions (Đã lọc trùng): [['Song Hành Xa lộ Hà Nội', 'Xa Lộ Hà Nội'], ['Xa Lộ Hà Nội', 'Quốc lộ 1'], ['Xa Lộ Hà Nội', 'Mai Chí Thọ']]


### Bước 2: Chạy FP-Growth và Trích xuất Luật (Rules)

Thuật toán AR yêu cầu đầu vào là một ma trận Boolean (True/False).

Việc thiết lập các tham số min_support và min_threshold cần được tinh chỉnh (fine-tune) dựa trên kích thước thực tế của tập dữ liệu.

In [2]:
# Huấn luyện FP-GROWTH
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_)

# min_support = 0.04 (phải xuất hiện cùng nhau ít nhất ~14 lần trên tổng 348 khung giờ)
# max_len = 3 (Chỉ tìm các tổ hợp gồm tối đa 3 tuyến đường để tránh bùng nổ tổ hợp)
frequent_itemsets = fpgrowth(df_transactions, min_support=0.04, use_colnames=True, max_len=3)
print(f"Tổ hợp phổ biến tìm được (min_support >= 0.04): {len(frequent_itemsets):,}")

if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

    # Lọc luật: Cần độ tin cậy >= 0.5
    strong_rules = rules[rules['confidence'] >= 0.5].copy()

    # Chỉ lấy các luật có đúng 1 con đường nguyên nhân (Antecedents độ dài = 1)
    strong_rules['antecedent_len'] = strong_rules['antecedents'].apply(lambda x: len(x))
    strong_rules = strong_rules[strong_rules['antecedent_len'] == 1]

    # Sắp xếp theo Support (Mức độ phổ biến) sau đó mới đến Lift (Độ liên kết)
    strong_rules = strong_rules.sort_values(by=['support', 'lift'], ascending=[False, False])

    print(f"Số lượng quy luật mạnh thực tế sau khi lọc: {len(strong_rules):,}")

    if len(strong_rules) > 0:
        strong_rules['antecedents_str'] = strong_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
        strong_rules['consequents_str'] = strong_rules['consequents'].apply(lambda x: ', '.join(list(x)))
        final_rules = strong_rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']]

        print("\nTop chuỗi ùn tắt có ý nghĩa cao nhất:")
        print(final_rules.head(10).to_string(index=False))

        final_rules.to_csv('traffic_rules_optimized.csv', index=False)

        # Lưu file
        final_rules.to_csv('traffic_association_rules_weekday.csv', index=False)
        print("\nĐã lưu kết quả xuất sắc nhất ra file: 'traffic_association_rules_weekday.csv'")
    else:
        print("\nKhông có luật nào đạt ngưỡng Confidence >= 0.4. Hãy thử giảm min_support xuống 0.005 hoặc giảm confidence xuống 0.3")
else:
    print("\nKhông tìm thấy tổ hợp phổ biến nào. Tập dữ liệu quá phân tán, hãy giảm min_support xuống 0.005")

Tổ hợp phổ biến tìm được (min_support >= 0.04): 64
Số lượng quy luật mạnh thực tế sau khi lọc: 33

Top chuỗi ùn tắt có ý nghĩa cao nhất:
       antecedents_str   consequents_str  support  confidence     lift
         Tô Hiến Thành    Lý Thường Kiệt 0.134948    0.812500 2.194509
     Nguyễn Ðình Chiểu       Trương Định 0.110727    0.666667 4.281481
           Trương Định Nguyễn Ðình Chiểu 0.110727    0.711111 4.281481
           Trương Định Nguyễn Bỉnh Khiêm 0.086505    0.555556 5.351852
     Nguyễn Bỉnh Khiêm       Trương Định 0.086505    0.833333 5.351852
               Lê Khôi    Lý Thường Kiệt 0.083045    0.888889 2.400831
Hẻm 373 Lý Thường Kiệt    Lý Thường Kiệt 0.076125    0.880000 2.376822
     Nguyễn Bỉnh Khiêm Nguyễn Ðình Chiểu 0.062284    0.600000 3.612500
         Lạc Long Quân    Lý Thường Kiệt 0.058824    0.944444 2.550883
         Hoàng Văn Thụ    Lý Thường Kiệt 0.058824    0.680000 1.836636

Đã lưu kết quả xuất sắc nhất ra file: 'traffic_association_rules_weekday.csv'


### Nhận xét:

* Nhận diện điểm nghẽn mạng lưới ở khu vực lõi: Thuật toán đã phát hiện ra sự sụp đổ dây chuyền mang tính nhân quả cực mạnh tại cụm giao thông trung tâm: Trương Định <-> Nguyễn Đình Chiểu <-> Nguyễn Bỉnh Khiêm. Hệ số Lift tại cụm này tăng đột biến (đạt từ 4.28 đến 5.35), cho thấy mối quan hệ phụ thuộc lẫn nhau rất chặt chẽ. Điển hình, khi Nguyễn Bỉnh Khiêm ùn tắc, xác suất kéo theo Trương Định tê liệt lên tới 83.3%. Điều này phản ánh rõ đặc tính dễ bị tổn thương của mạng lưới đường ô bàn cờ: một mắt xích đứt gãy sẽ lập tức phân bổ lại áp lực và đánh sập các tuyến song song hoặc giao cắt lân cận.

* Hiệu ứng tràn bờ trên trục động lực: Lý Thường Kiệt là một trục giao thông hứng chịu hậu quả rõ rệt nhất từ các tuyến đường nhánh. Dữ liệu chứng minh rành mạch rằng luồng phương tiện từ các hẻm/đường nhỏ xả ra là nguyên nhân trực tiếp gây tắc nghẽn trục chính. Cụ thể, khi Lạc Long Quân, Lê Khôi, hay Hẻm 373 Lý Thường Kiệt kẹt xe, nó sẽ kéo theo sự sụp đổ năng lực thông hành của toàn tuyến Lý Thường Kiệt với độ tin cậy áp đảo: lần lượt là 94.4%, 88.8% và 88.0%. Tổ hợp Tô Hiến Thành -> Lý Thường Kiệt cũng là chuỗi phổ biến nhất tập dữ liệu (Support cao nhất ~13.5%).

* Giá trị ứng dụng trong Quy hoạch & Dự báo : Bộ 33 quy luật cốt lõi này vượt qua ranh giới của thống kê đơn thuần để cung cấp các bằng chứng định lượng về tính liên kết không gian. Đây là nguồn đặc trưng giá trị cao để làm đầu vào cho các mô hình Deep Learning dự báo lan truyền ùn tắc đồ thị (Graph Neural Networks). Trong quy hoạch TOD (Transit-Oriented Development), phát hiện này chỉ điểm chính xác các vùng đỏ cần tái định tuyến xe buýt nhằm tránh đi xuyên qua cụm lõi dễ sụp đổ (cụm Trương Định), đồng thời đòi hỏi phải có giải pháp kiểm soát xung đột giao thông tại các nút giao nhánh đổ ra trục chính (dọc tuyến Lý Thường Kiệt).